In [1]:
import sys
sys.path.append("/anvme/workspace/v120bb18-match/")
import os
import torch
import yaml
from dotmap import DotMap
from dataset import CHOLEC80
from pipelines.odometry.odometry import OdometryPipeline
from pipelines.depth.depth import DepthPipeline
from pipelines.matching.matching import MatchingPipeline
from pipelines.depth.tsdf import TSDFVolume
import matplotlib.pyplot as plt
from utilities.visualization import viewComparePixelMatches,log_rerun_camera
import numpy as np
%load_ext autoreload
%autoreload 2

/anvme/workspace/v120bb18-match/.venv/lib/python3.12/site-packages/torch_sgld/lr_scheduler.py:13: SyntaxWarning: invalid escape sequence '\e'
  \epsilon_t = a(b + t)^{-\gamma}
/anvme/workspace/v120bb18-match/.venv/lib/python3.12/site-packages/torch_sgld/lr_scheduler.py:66: SyntaxWarning: invalid escape sequence '\l'
  \alpha_k = \frac{\alpha_0}{2} \left[ \cos{\frac{\pi\mod{k-1, \ceil{K/M}}}{\ceil{K/M}}} \right]


In [2]:
base = "/anvme/workspace/v120bb18-unreflectanything/datasets/EndoSynth/EndoSynth/samples"
data = np.load(os.path.join(base,'002340.npz'))

# List all arrays stored in the file
print("Arrays in file:", data.files)
from utilities.visualization import rgb


Arrays in file: ['rgb', 'depth', 'seg']


In [6]:
from EndoSynth.endosynth.models import load
from EndoSynth.endosynth.utils import depth2rgb
import numpy as np
from PIL import Image

model = load("dav1")
# model = load("dav2")
# model = load("endodac")
# model = load("midas")
rgbimage = data["rgb"]
depth = model.infer(rgbimage)
rgb(depth)
# Image.fromarray(depth2rgb(depth, 0.02, 0.20).astype(np.uint8)).save("out.png")

ModuleNotFoundError: No module named 'depth_anything'

In [4]:
CONFIG_PATH = 'config_train.yaml'
with open(CONFIG_PATH, 'r') as f:
    config_yaml = yaml.safe_load(f)
    config_parameters = config_yaml['parameters']
    config_training_dict = {k: v.get('value') for k, v in config_parameters.items() if v is not None}
    config = DotMap(config_training_dict)


In [14]:
dataset = CHOLEC80(
    path="/anvme/workspace/v120bb18-unreflectanything/datasets/CHOLEC80/videos",
    # vids=["v33"],
    exclude=["val_"],
    frameskip=[32],
    fps = 32,
    random_pose=False,
    height=config.IMAGE_HEIGHT,
    width=config.IMAGE_WIDTH,
    with_depth=False,
    unit_translation=False,
)
len(dataset)


161742

In [ ]:

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1)
iloader = iter(dataloader)
sample = next(iloader)
# Print sample keys and shapes
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'{k}: Tensor of shape {list(v.shape)}')
    else:
        print(f'{k}: {type(v)}')
